# Notebook 5: Macro Analysis

So far we have shown that the value premium exists, has weakened over time, 
and generates alpha beyond the Fama-French factors for BM.

Now we ask: WHY has value underperformed in recent years?

One popular explanation is interest rates. When interest rates fall, investors 
are willing to pay more for future growth, which benefits growth stocks more 
than value stocks. Growth companies like Amazon and Google derive most of their 
value from cash flows far in the future. When you discount those future cash flows 
at a lower rate, they become worth much more today.

We test this by downloading real interest rate and market volatility data from 
FRED (the Federal Reserve's data repository) and running regressions to see if 
these macro variables explain the value spread.

Variables we will use:
- **GS10:** 10-year Treasury yield — our main interest rate measure
- **VIXCLS:** The VIX — measures market fear/volatility

In [1]:
!pip install fredapi


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from fredapi import Fred

from dotenv import load_dotenv
import os

plt.style.use('seaborn-v0_8-whitegrid')

bm_wide = pd.read_csv('bm_wide.csv', index_col='date', parse_dates=True)
ep_wide = pd.read_csv('ep_wide.csv', index_col='date', parse_dates=True)

bm_wide.index = bm_wide.index + pd.offsets.MonthEnd(0)
ep_wide.index = ep_wide.index + pd.offsets.MonthEnd(0)


### Download Macro Data from FRED

We download two variables:
- 10-year Treasury yield (GS10): when this falls, growth stocks tend to win
- VIX (VIXCLS): measures market fear — high VIX periods often hurt value stocks

In [8]:
load_dotenv()

fred = Fred(api_key=os.getenv('FRED_API_KEY'))

gs10 = fred.get_series('GS10', start='1963-01-01')
vix  = fred.get_series('VIXCLS', start='1963-01-01')

gs10_monthly = gs10.resample('ME').last()
vix_monthly  = vix.resample('ME').last()

macro = pd.DataFrame({
    'gs10': gs10_monthly,
    'vix':  vix_monthly
})

print("Date range:", macro.index.min(), "to", macro.index.max())
print(macro.tail())

ValueError: You need to set a valid API key. You can set it in 3 ways:
pass the string with api_key, or set api_key_file to a
file with the api key in the first line, or set the
environment variable 'FRED_API_KEY' to the value of your
api key. You can sign up for a free api key on the Fred
website at http://research.stlouisfed.org/fred2/